In [18]:
import numpy as np 
import pandas as pd 
from sklearn.feature_extraction.text import TfidfVectorizer

In [19]:
def import_data(location: str) -> pd.DataFrame:
    
    if location.endswith('.json'):
        df = pd.read_json(location, lines = True)  
    elif location.endswith('.csv'):
        df = pd.read_csv(location)
    else:
        raise ValueError('Unusable file type')  
    return df

In [20]:
vet_business = import_data('/kaggle/input/datasets/micahluftig/yelp-veterinary-data/yelp_veterinary_business.csv')
vet_check_in = import_data('/kaggle/input/datasets/micahluftig/yelp-veterinary-data/yelp_veterinary_check_in.csv')
vet_reviews  = import_data('/kaggle/input/datasets/micahluftig/yelp-veterinary-data/yelp_veterinary_reviews.csv')
vet_tip      = import_data('/kaggle/input/datasets/micahluftig/yelp-veterinary-data/yelp_veterinary_tip.csv')
vet_user     = import_data('/kaggle/input/datasets/micahluftig/yelp-veterinary-data/yelp_veterinary_user.csv')

In [21]:
# 1. Initialize the vectorizer
tfidf = TfidfVectorizer(
    max_features = 5000,      
    min_df = 3,                
    max_df = 0.5,               
    ngram_range = (1, 2)       
)

tfidf_matrix = tfidf.fit_transform(vet_reviews['clean_text'])

print(tfidf_matrix.shape)

(33512, 5000)


In [22]:
feature_names = tfidf.get_feature_names_out()

avg_scores = np.asarray(tfidf_matrix.mean(axis=0)).flatten()

top_n = 30
top_indices = avg_scores.argsort()[-top_n:][::-1]

for i in top_indices:
    print(feature_names[i], round(avg_scores[i], 4))

care 0.0303
great 0.0237
time 0.0234
place 0.0204
love 0.0169
years 0.0169
recommend 0.0164
never 0.0159
friendly 0.0158
best 0.0157
hospital 0.0154
good 0.0153
clinic 0.0153
could 0.0151
well 0.015
called 0.0144
told 0.0143
office 0.0136
appointment 0.0133
first 0.0129
see 0.0129
visit 0.0128
ive 0.0128
service 0.0127
know 0.0126
new 0.0125
much 0.0125
said 0.0123
make 0.0122
kind 0.0122


In [23]:
low = tfidf.transform(vet_reviews.loc[vet_reviews['stars'] <= 2, 'clean_text'])
high = tfidf.transform(vet_reviews.loc[vet_reviews['stars'] >= 4, 'clean_text'])

low_avg = np.asarray(low.mean(axis = 0)).flatten()
high_avg = np.asarray(high.mean(axis = 0)).flatten()
diff = low_avg - high_avg  

In [24]:
top_negative = diff.argsort()[-20:][::-1]
print("Words associated with LOW ratings:")
for i in top_negative:
    print(feature_names[i], round(diff[i], 4))

Words associated with LOW ratings:
told 0.034
said 0.0278
money 0.0172
rude 0.016
asked 0.0157
never 0.0155
call 0.0136
called 0.0135
another 0.013
could 0.0128
later 0.0094
pay 0.0094
left 0.0093
horrible 0.0091
blood 0.0089
still 0.0089
charged 0.0087
worst 0.0087
business 0.0085
minutes 0.0083


In [25]:
top_positive = diff.argsort()[:20]  
print("Words associated with HIGH ratings:")
for i in top_positive:
    print(feature_names[i], round(diff[i], 4))

Words associated with HIGH ratings:
great -0.0259
love -0.0177
friendly -0.0174
best -0.0161
care -0.015
highly -0.0143
amazing -0.0143
wonderful -0.0142
thank -0.0134
recommend -0.013
highly recommend -0.013
helpful -0.012
kind -0.0119
knowledgeable -0.0107
compassionate -0.0098
everyone -0.0096
excellent -0.0094
professional -0.0093
happy -0.0089
thorough -0.008


In [26]:
top_n = 20
top_indices = np.concatenate([top_negative, top_positive])  

results = pd.DataFrame({
    'word': [feature_names[i] for i in top_indices],
    'diff': [diff[i] for i in top_indices]
})
results = results.sort_values('diff')  
results.to_csv('tfidf_diff_top_words.csv', index = False)